# NB01 · Inventario y auditoría del dataset de palma africana

Este notebook inspecciona el contenido real descargado desde Zenodo antes de elegir el diseño experimental.

**Entorno recomendado:** CPU, sin acelerador  
**Tiempo estimado:** 10–20 minutos si el ZIP ya fue extraído; 15–30 minutos en la primera ejecución.

Comprueba:

- estructura de carpetas y extensiones;
- número de imágenes por clase y partición;
- imágenes dañadas;
- dimensiones y modos de color;
- duplicados exactos mediante SHA-256;
- presencia o ausencia de anotaciones de detección.

> La auditoría define si el proyecto se implementará como clasificación de madurez con YOLO11-cls o si será necesario generar cajas para detección.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
from collections import Counter
import hashlib
import json
import re
import shutil
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageOps

PROJECT_ROOT = Path('/content/drive/MyDrive/PALMA_AFRICANA_YOLO')
RAW_DIR = PROJECT_ROOT / '01_DATOS_ORIGINALES'
EXTRACT_DIR = RAW_DIR / 'zenodo_11114885_extracted'
RESULTS_DIR = PROJECT_ROOT / '06_RESULTADOS'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

zip_candidates = sorted(RAW_DIR.glob('*.zip'))
if not zip_candidates:
    raise FileNotFoundError('No se encontró el ZIP. Ejecute primero NB00.')
ZIP_PATH = zip_candidates[0]
print('ZIP:', ZIP_PATH)
print('Tamaño:', f'{ZIP_PATH.stat().st_size/1024**3:.3f} GB')


In [ ]:
with zipfile.ZipFile(ZIP_PATH) as archive:
    members = archive.infolist()
    print('Entradas dentro del ZIP:', len(members))
    ext_counts = Counter(Path(m.filename).suffix.lower() or '[sin extensión]' for m in members if not m.is_dir())
    print('Extensiones más frecuentes:')
    for ext, count in ext_counts.most_common(15):
        print(f'  {ext}: {count}')

    marker = EXTRACT_DIR / '.extraction_complete'
    if not marker.exists():
        EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
        archive.extractall(EXTRACT_DIR)
        marker.write_text('Extracción completada', encoding='utf-8')
        print('Extracción completada:', EXTRACT_DIR)
    else:
        print('La extracción ya estaba completada:', EXTRACT_DIR)


In [ ]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
LABEL_EXTS = {'.txt', '.xml', '.json'}

images = sorted(p for p in EXTRACT_DIR.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS)
possible_labels = sorted(p for p in EXTRACT_DIR.rglob('*') if p.is_file() and p.suffix.lower() in LABEL_EXTS)

print('Imágenes encontradas:', len(images))
print('Archivos que podrían contener anotaciones:', len(possible_labels))
print('Ejemplos de rutas:')
for p in images[:10]:
    print('-', p.relative_to(EXTRACT_DIR))


In [ ]:
CLASSES = ['damaged', 'empty', 'overripe', 'unripe', 'ripe']

def infer_class(path):
    text = str(path.relative_to(EXTRACT_DIR)).lower().replace('_', ' ')
    for label in CLASSES:
        if label in text:
            return label
    return 'unknown'

def infer_split(path):
    text = str(path.relative_to(EXTRACT_DIR)).lower()
    if 'train' in text:
        return 'train_original'
    if 'test' in text:
        return 'test_original'
    return 'unassigned'

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        while True:
            chunk = stream.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

rows = []
for idx, path in enumerate(images, 1):
    row = {
        'relative_path': str(path.relative_to(EXTRACT_DIR)),
        'filename': path.name,
        'split_inferred': infer_split(path),
        'class_inferred': infer_class(path),
        'bytes': path.stat().st_size,
        'sha256': sha256_file(path),
        'valid': True,
        'error': '',
        'width': np.nan,
        'height': np.nan,
        'mode': '',
    }
    try:
        with Image.open(path) as im:
            im.verify()
        with Image.open(path) as im:
            row['width'], row['height'] = im.size
            row['mode'] = im.mode
    except Exception as exc:
        row['valid'] = False
        row['error'] = repr(exc)
    rows.append(row)
    if idx % 50 == 0 or idx == len(images):
        print(f'Auditadas: {idx}/{len(images)}', end='\r')
print()

inventory = pd.DataFrame(rows)
inventory.to_csv(RESULTS_DIR / 'inventario_imagenes.csv', index=False)
display(inventory.head())


In [ ]:
summary = (
    inventory.groupby(['split_inferred', 'class_inferred'], dropna=False)
    .size()
    .rename('n')
    .reset_index()
)
summary.to_csv(RESULTS_DIR / 'resumen_clases_splits.csv', index=False)
display(summary)

print('Imágenes válidas:', int(inventory['valid'].sum()))
print('Imágenes dañadas:', int((~inventory['valid']).sum()))
print('Clases no reconocidas:', int((inventory['class_inferred'] == 'unknown').sum()))
print('Sin partición reconocida:', int((inventory['split_inferred'] == 'unassigned').sum()))

duplicate_groups = inventory.groupby('sha256').filter(lambda g: len(g) > 1).sort_values('sha256')
duplicate_groups.to_csv(RESULTS_DIR / 'duplicados_exactos.csv', index=False)
print('Archivos dentro de grupos duplicados exactos:', len(duplicate_groups))
if len(duplicate_groups):
    display(duplicate_groups[['relative_path', 'split_inferred', 'class_inferred', 'sha256']].head(30))


In [ ]:
pivot = summary.pivot(index='class_inferred', columns='split_inferred', values='n').fillna(0)
ax = pivot.plot(kind='bar', figsize=(10, 5), color=['#1f77b4', '#ff7f0e', '#9e9e9e'])
ax.set_title('Distribución de imágenes por clase y partición original')
ax.set_xlabel('Clase')
ax.set_ylabel('Número de imágenes')
ax.grid(axis='y', alpha=0.25)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'distribucion_clases_splits.png', dpi=220, bbox_inches='tight')
plt.show()

valid = inventory[inventory['valid']].copy()
plt.figure(figsize=(7, 5))
plt.scatter(valid['width'], valid['height'], alpha=0.55, s=28, color='#2e7d32')
plt.title('Resolución de las imágenes')
plt.xlabel('Ancho (px)')
plt.ylabel('Alto (px)')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'resoluciones_imagenes.png', dpi=220, bbox_inches='tight')
plt.show()


In [ ]:
fig, axes = plt.subplots(len(CLASSES), 4, figsize=(12, 14))
for row_idx, label in enumerate(CLASSES):
    candidates = inventory[(inventory['class_inferred'] == label) & inventory['valid']]
    sample = candidates.sample(min(4, len(candidates)), random_state=2026) if len(candidates) else candidates
    for col_idx, ax in enumerate(axes[row_idx]):
        ax.axis('off')
        if col_idx < len(sample):
            item = sample.iloc[col_idx]
            path = EXTRACT_DIR / item['relative_path']
            with Image.open(path) as im:
                try:
                    display_image = ImageOps.exif_transpose(im).convert('RGB')
                except Exception:
                    # Algunos JPEG del conjunto contienen EXIF no estándar.
                    # La imagen sigue siendo válida; se muestra sin aplicar orientación EXIF.
                    display_image = im.convert('RGB')
                ax.imshow(np.asarray(display_image))
            ax.set_title(f'{label} · {item["split_inferred"]}', fontsize=9)
plt.suptitle('Muestra visual por clase', fontsize=14)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'muestra_visual_clases.png', dpi=220, bbox_inches='tight')
plt.show()


In [ ]:
label_like = [p for p in possible_labels if p.name.lower() not in {'readme.txt', 'license.txt'}]
decision = {
    'n_images': int(len(inventory)),
    'n_valid': int(inventory['valid'].sum()),
    'n_corrupt': int((~inventory['valid']).sum()),
    'n_exact_duplicate_rows': int(len(duplicate_groups)),
    'classes_detected': sorted(inventory.loc[inventory['class_inferred'] != 'unknown', 'class_inferred'].unique().tolist()),
    'possible_annotation_files': [str(p.relative_to(EXTRACT_DIR)) for p in label_like[:100]],
    'recommended_initial_task': 'classification',
    'reason': 'Las etiquetas publicadas corresponden a clases de madurez; no se asumen cajas hasta comprobar anotaciones de detección.'
}
(RESULTS_DIR / 'decision_inicial_dataset.json').write_text(
    json.dumps(decision, ensure_ascii=False, indent=2), encoding='utf-8'
)
print(json.dumps(decision, ensure_ascii=False, indent=2))


## Criterio para continuar

- Si el inventario confirma únicamente etiquetas por clase, el siguiente notebook preparará una partición **leakage-safe** para **YOLO11-cls**.
- Si aparecen cajas o anotaciones compatibles, se evaluará el flujo de **YOLO11 detect**.
- No se usarán automáticamente las divisiones publicadas sin revisar duplicados y posible dependencia entre imágenes del mismo racimo.

Archivos producidos en `06_RESULTADOS`:

- `inventario_imagenes.csv`
- `resumen_clases_splits.csv`
- `duplicados_exactos.csv`
- `distribucion_clases_splits.png`
- `resoluciones_imagenes.png`
- `muestra_visual_clases.png`
- `decision_inicial_dataset.json`
